# 12 — Broadcast Joins & Broadcast Variables

**What you will learn:**
- How Spark chooses join strategies (Sort-Merge vs Broadcast Hash Join)
- `broadcast()` hint — force a small table to broadcast
- `spark.sql.autoBroadcastJoinThreshold` — auto-broadcast size limit
- SQL `/*+ BROADCAST */` hint
- Broadcast variables (`sc.broadcast`) for lookup dictionaries
- When NOT to use broadcast
- Reading the query plan to confirm a broadcast join

## 1. Two Main Join Strategies

| Strategy | When Used | How | Shuffle Cost |
|---|---|---|---|
| **Sort-Merge Join** | Both tables large | Shuffle both sides, sort, merge | High — both sides shuffled |
| **Broadcast Hash Join** | One table small | Copy small table to every executor | Zero — large side not shuffled |

**Broadcast is dramatically faster** for small-large joins because it eliminates the shuffle on the large side.

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast
import time

# Large fact table (500k rows)
orders = spark.createDataFrame(
    [(i, i % 1000, float(i * 13 % 500)) for i in range(1, 500001)],
    ["order_id", "customer_id", "amount"]
)

# Small dimension table (1k rows — ideal for broadcast)
customers = spark.createDataFrame(
    [(i, f"Customer_{i}", ["US","UK","DE","FR"][i % 4]) for i in range(1, 1001)],
    ["customer_id", "name", "country"]
)

print(f"Orders:    {orders.count():,} rows")
print(f"Customers: {customers.count():,} rows")

## 2. Default Join — Sort-Merge (both sides shuffled)

In [ ]:
# No hint — Spark chooses Sort-Merge if customers exceeds auto-broadcast threshold
df_smj = orders.join(customers, on="customer_id", how="inner")
df_smj.explain(mode="formatted")  # Look for: SortMergeJoin

## 3. Broadcast Join — Explicit Hint

In [ ]:
# Force broadcast of the small customers table
df_bcast = orders.join(broadcast(customers), on="customer_id", how="inner")
df_bcast.explain(mode="formatted")  # Look for: BroadcastHashJoin

# Benchmark
t0 = time.time(); orders.join(customers, on="customer_id").count(); smj = time.time()-t0
t0 = time.time(); orders.join(broadcast(customers), on="customer_id").count(); bj = time.time()-t0

print(f"Sort-Merge Join : {smj:.2f}s")
print(f"Broadcast Join  : {bj:.2f}s")
print(f"Speedup         : {smj/bj:.1f}x")

## 4. Auto-Broadcast Threshold

Spark **automatically broadcasts** tables smaller than `spark.sql.autoBroadcastJoinThreshold` (default **10 MB**).
Raise it to broadcast larger dimension tables; set to `-1` to disable auto-broadcast.

In [ ]:
print("Current threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Raise to 50 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)
print("Raised to 50 MB")

# Disable auto-broadcast (force Sort-Merge always — useful for testing)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
print("Auto-broadcast disabled")

# Reset to default
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
print("Reset to 10 MB default")

## 5. SQL Broadcast Hint

Apply the broadcast hint inside a SQL string using `/*+ BROADCAST(alias) */`.

In [ ]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

df_sql = spark.sql('''
    SELECT /*+ BROADCAST(c) */
        o.order_id, o.amount, c.name, c.country
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
''')

df_sql.explain(mode="simple")
df_sql.limit(5).show()

## 6. Broadcast Variables — Ship Any Object to Executors

Broadcast variables ship any Python object (dict, list, ML model) to **every executor once** and pin it in memory.
Ideal for: lookup dictionaries, country codes, ML scoring models.

In [ ]:
# Lookup dictionary: country code → full name
country_map = {"US": "United States", "UK": "United Kingdom",
               "DE": "Germany",       "FR": "France"}

# Broadcast once to all executors
bc_country = spark.sparkContext.broadcast(country_map)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def expand_country(code):
    return bc_country.value.get(code, "Unknown")   # bc.value accesses the dict on executor

customers.withColumn("country_full", expand_country(F.col("country"))).show(5)

# Release when done
bc_country.unpersist()

## 7. When NOT to Broadcast

- Table is **large** (hundreds of MB+) — too expensive to copy to every executor
- **Many concurrent queries** — each holds a copy in executor memory, pressure mounts
- Table **changes frequently** — broadcast copies become stale between queries

## Key Takeaways

| Concept | Detail |
|---|---|
| Default join | Sort-Merge — both sides shuffled (expensive for large tables) |
| Broadcast join | Small table copied to each executor — no shuffle on large side |
| Explicit hint | `broadcast(df)` in Python or `/*+ BROADCAST(alias) */` in SQL |
| Auto threshold | `spark.sql.autoBroadcastJoinThreshold` = 10 MB default |
| Broadcast variable | `sc.broadcast(obj)` — ship any Python object once |
| Always | Release with `bc.unpersist()` when done |